# NeuroTranslate AR→EN Notebook

## 1. Setup

In [3]:
import re
import os
import math
import random
import tempfile
import unicodedata
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

try:
    import sentencepiece as spm
except ImportError as exc:
    raise ImportError("Please install sentencepiece first: pip install sentencepiece") from exc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

HIDDEN_SIZE = 384
EMBED_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.3
MAX_LEN = 32
N_EPOCHS = 30
LEARNING_RATE = 5e-4
MIN_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 2e-5
TEACHER_FORCE_START = 0.95
TEACHER_FORCE_END = 0.25
CLIP = 1.0
BATCH_SIZE = 64
PATIENCE = 6
LABEL_SMOOTHING = 0.05

PAD_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
UNK_TOKEN = 3

SRC_SPM_VOCAB = 3200
TGT_SPM_VOCAB = 2400
SPM_DIR = "spm_models"

Device: cpu


## 2. Data Loading & Preprocessing

In [4]:
import urllib.request

RAW_URL = ("https://raw.githubusercontent.com/SamirMoustafa/"
           "nmt-with-attention-for-ar-to-en/master/ara_.txt")
DATA_PATH = "ara_.txt"

if not os.path.isfile(DATA_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(RAW_URL, DATA_PATH)
    print("Dataset downloaded.")
else:
    print(f"Using local dataset: {DATA_PATH}")

Dataset downloaded.


In [5]:
def unicode_to_ascii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def normalize_english(s):
    s = unicode_to_ascii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?']+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

ARABIC_DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")

def normalize_arabic(s):
    s = unicodedata.normalize("NFKC", s).strip()
    s = s.replace("ـ", "")
    s = ARABIC_DIACRITICS_RE.sub("", s)

    s = re.sub(r"[إأٱآ]", "ا", s)
    s = re.sub(r"ى", "ي", s)
    s = re.sub(r"ؤ", "و", s)
    s = re.sub(r"ئ", "ي", s)

    s = re.sub(r"([؟!?.,])", r" \1 ", s)
    s = re.sub(r"[^\u0600-\u06FF\s؟!?.,]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def contains_arabic(text):
    return bool(re.search(r"[\u0600-\u06FF]", text))

def read_pairs(path):
    lines = open(path, encoding="utf-8").read().strip().split("\n")
    pairs = []
    for line in lines:
        parts = line.split("\t")
        if len(parts) < 2:
            continue

        a, b = parts[0], parts[1]
        if contains_arabic(a):
            src_ar, tgt_en = a, b
        else:
            src_ar, tgt_en = b, a

        src_ar = normalize_arabic(src_ar)
        tgt_en = normalize_english(tgt_en)

        if src_ar and tgt_en:
            pairs.append((src_ar, tgt_en))
    return pairs

def filter_pair(pair):
    src_len = len(pair[0].split())
    tgt_len = len(pair[1].split())
    return 2 <= src_len <= MAX_LEN and 2 <= tgt_len <= MAX_LEN

all_pairs = read_pairs(DATA_PATH)
pairs = [p for p in all_pairs if filter_pair(p)]
random.shuffle(pairs)

print(f"Total pairs (raw): {len(all_pairs)}")
print(f"Total pairs (filtered): {len(pairs)}")
print()
print("Sample Arabic -> English pairs:")
for src_ar, tgt_en in pairs[:3]:
    print(f"  AR: {src_ar}")
    print(f"  EN: {tgt_en}")
    print()

Total pairs (raw): 10742
Total pairs (filtered): 10731

Sample Arabic -> English pairs:
  AR: صباح الخير . انه وقت الاستيقاظ .
  EN: good morning . it's time to wake up .

  AR: انا لا الومك .
  EN: i don't blame you .

  AR: مهما كان توم مشغولا، لم ينسي ابدا كتابة رسالة الي امه مرة في الاسبوع علي الاقل .
  EN: no matter how busy tom gets he never forgets to write an email to his mother at least once a week .



## 3. Vocabulary Construction

In [6]:
n = len(pairs)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

os.makedirs(SPM_DIR, exist_ok=True)

def train_sentencepiece_model(texts, model_prefix, vocab_size, character_coverage=1.0):
    with tempfile.NamedTemporaryFile("w", delete=False, encoding="utf-8") as tmp:
        for line in texts:
            tmp.write(line.strip() + "\n")
        temp_path = tmp.name

    spm.SentencePieceTrainer.Train(
        input=temp_path,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=character_coverage,
        normalization_rule_name="identity",
        split_digits=False,
        pad_id=PAD_TOKEN,
        bos_id=SOS_TOKEN,
        eos_id=EOS_TOKEN,
        unk_id=UNK_TOKEN,
        pad_piece="<PAD>",
        bos_piece="<SOS>",
        eos_piece="<EOS>",
        unk_piece="<UNK>",
        hard_vocab_limit=False,
        train_extremely_large_corpus=False,
    )
    os.remove(temp_path)


class SubwordTokenizer:
    def __init__(self, model_path, name):
        self.name = name
        self.sp = spm.SentencePieceProcessor(model_file=model_path)
        self.n_words = self.sp.get_piece_size()

    def encode(self, text):
        return self.sp.encode(text, out_type=int)

    def decode(self, token_ids):
        if not token_ids:
            return ""
        return self.sp.decode(token_ids)

    def pieces(self, text):
        return self.sp.encode(text, out_type=str)


src_sp_prefix = os.path.join(SPM_DIR, "ar_unigram")
tgt_sp_prefix = os.path.join(SPM_DIR, "en_unigram")
src_model_path = f"{src_sp_prefix}.model"
tgt_model_path = f"{tgt_sp_prefix}.model"

if not os.path.isfile(src_model_path):
    train_sentencepiece_model(
        [src for src, _ in train_pairs],
        src_sp_prefix,
        SRC_SPM_VOCAB,
        character_coverage=0.9995,
    )

if not os.path.isfile(tgt_model_path):
    train_sentencepiece_model(
        [tgt for _, tgt in train_pairs],
        tgt_sp_prefix,
        TGT_SPM_VOCAB,
        character_coverage=1.0,
    )

src_tokenizer = SubwordTokenizer(src_model_path, "arabic_sp")
tgt_tokenizer = SubwordTokenizer(tgt_model_path, "english_sp")

print(f"Arabic tokenizer vocab size : {src_tokenizer.n_words}")
print(f"English tokenizer vocab size: {tgt_tokenizer.n_words}")
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")

Arabic tokenizer vocab size : 3200
English tokenizer vocab size: 2400
Train: 8584 | Val: 1073 | Test: 1074


In [7]:
def sentence_to_tensor(tokenizer, sentence, add_sos=False):
    ids = tokenizer.encode(sentence)
    if add_sos:
        ids = [SOS_TOKEN] + ids
    ids.append(EOS_TOKEN)
    return torch.tensor(ids, dtype=torch.long)

def pair_to_tensors(pair):
    src, tgt = pair
    src_tensor = sentence_to_tensor(src_tokenizer, src, add_sos=False)
    tgt_tensor = sentence_to_tensor(tgt_tokenizer, tgt, add_sos=True)
    return src_tensor, tgt_tensor

## 4. Model Configuration

## 5. Training

## 6. Evaluation

## 7. Inference Examples

## 8. Error Analysis

## 9. Notes & Next Steps